## Create environment

In [1]:
%%bash
set -e

cd /content
pwd

curl -fL https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/micromamba.tar.bz2
ls -lh /content/micromamba.tar.bz2

tar -xvjf /content/micromamba.tar.bz2 -C /usr/local/bin --strip-components=1 bin/micromamba

which micromamba
micromamba --version

/content
-rw-r--r-- 1 root root 6.4M Mar 23 15:20 /content/micromamba.tar.bz2
bin/micromamba
/usr/local/bin/micromamba
2.5.0


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  4095    0  4095    0     0   3494      0 --:--:--  0:00:01 --:--:--  3494
100 6493k  100 6493k    0     0  3213k      0  0:00:02  0:00:02 --:--:-- 39.1M


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/GWIT2"

# cache persistenti
HF_HOME = f"{PROJECT_ROOT}/.cache/huggingface"
PIP_CACHE_DIR = f"{PROJECT_ROOT}/.cache/pip"

# runtime locale
ENV_PATH = "/content/micromamba/envs/gwit"
LOCAL_ROOT = "/content/gwit_runtime"
LOCAL_LATENTS = f"{LOCAL_ROOT}/latents"

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(PIP_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)
os.makedirs(LOCAL_LATENTS, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("ENV_PATH     =", ENV_PATH)
print("HF_HOME      =", HF_HOME)

PROJECT_ROOT = /content/drive/MyDrive/GWIT2
ENV_PATH     = /content/micromamba/envs/gwit
HF_HOME      = /content/drive/MyDrive/GWIT2/.cache/huggingface


In [4]:
import os

os.environ["HF_HOME"] = HF_HOME
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{HF_HOME}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{HF_HOME}/datasets"
os.environ["PIP_CACHE_DIR"] = PIP_CACHE_DIR
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("HF_HOME =", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])
print("PIP_CACHE_DIR =", os.environ["PIP_CACHE_DIR"])

HF_HOME = /content/drive/MyDrive/GWIT2/.cache/huggingface
HF_DATASETS_CACHE = /content/drive/MyDrive/GWIT2/.cache/huggingface/datasets
PIP_CACHE_DIR = /content/drive/MyDrive/GWIT2/.cache/pip


In [5]:
acc_dir = f"{HF_HOME}/accelerate"
os.makedirs(acc_dir, exist_ok=True)

config_text = """compute_environment: LOCAL_MACHINE
distributed_type: NO
machine_rank: 0
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
"""

with open(f"{acc_dir}/default_config.yaml", "w") as f:
    f.write(config_text)

print("✔ accelerate default_config.yaml creato in", acc_dir)

✔ accelerate default_config.yaml creato in /content/drive/MyDrive/GWIT2/.cache/huggingface/accelerate


In [6]:
import os

if not os.path.isdir(ENV_PATH):
    print("Creo environment locale...")
    !micromamba create -y -p "{ENV_PATH}" python=3.9
    !micromamba run -p "{ENV_PATH}" pip install --upgrade pip
    !micromamba run -p "{ENV_PATH}" pip install ./diffusers/
    !micromamba run -p "{ENV_PATH}" pip install -r requirements.txt
    !micromamba run -p "{ENV_PATH}" pip install hf_transfer
else:
    print("Environment locale già presente in questa sessione.")

Creo environment locale...
[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   2%
conda-forge/noarch    19%[+] 0.3s
conda-forge/linux-64  12%
conda-forge/noarch    39%[+] 0.4s
conda-forge/linux-64  20%
conda-forge/noarch    55%[+] 0.5s
conda-forge/linux-64  26%
conda-forge/noarch    68%[+] 0.6s
conda-forge/linux-64  34%
conda-forge/noarch    86%conda-forge/noarch                                
[+] 0.7s
conda-forge/linux-64  42%[+] 0.8s
conda-forge/linux-64  50%[+] 0.9s
conda-forge/linux-64  55%[+] 1.0s
conda-forge/linux-64  64%[+] 1.1s
conda-forge/linux-64  67%[+] 1.2s
conda-forge/linux-64  67%[+] 1.3s
conda-forge/linux-64  67%[+] 1.4s
conda-forge/linux-64  67%[+] 1.5s
conda-forge/linux-64  67%[+] 1.6s
conda-forge/linux-64  67%[+] 1.7s
conda-forge/linux-64  67%[+] 1.8s
conda-forge/linux-64  67%[+] 1.9s
conda-forge/linux-64  67%[+] 2.0s
conda-forge/linux-64  67%[+] 2.1s
conda-forge/linux-64  70%[+] 2.2s
conda-forge/linux-64  88%[+] 2.3s


In [7]:
DRIVE_LATENTS = f"{PROJECT_ROOT}/data/latents"
LOCAL_LATENTS = f"{LOCAL_ROOT}/latents"

!mkdir -p "{LOCAL_LATENTS}"
!rsync -ah --delete --info=progress2 "{DRIVE_LATENTS}/" "{LOCAL_LATENTS}/"

          1.57G 100%   21.26MB/s    0:01:10 (xfr#12, to-chk=0/20)


In [8]:
DRIVE_CLIP_EMBEDS = f"{PROJECT_ROOT}/data/clip_embeds"
LOCAL_CLIP_EMBEDS = f"{LOCAL_ROOT}/clip_embeds"

!mkdir -p "{LOCAL_CLIP_EMBEDS}"
!rsync -ah --delete --info=progress2 "{DRIVE_CLIP_EMBEDS}/" "{LOCAL_CLIP_EMBEDS}/"

         28.54M 100%    5.44MB/s    0:00:05 (xfr#9, to-chk=0/20)


## Precompute latents

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit python3 precompute_latents.py \
  --model_name Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --batch_size 16 \
  --save_every 25 \
  --output_root data/latents

/content/drive/My Drive/GWIT2
[INFO] Loading full HF pool: train + validation + test
Generating test split: 100% 1987/1987 [00:02<00:00, 763.57 examples/s]
Generating train split: 100% 7959/7959 [00:13<00:00, 598.56 examples/s]
Generating validation split: 100% 1994/1994 [00:07<00:00, 263.73 examples/s]
[INFO] Full pool size: 11940 samples
[INFO] Loading VAE...
config.json: 100% 553/553 [00:00<00:00, 344kB/s]
diffusion_pytorch_model.safetensors: 100% 335M/335M [00:09<00:00, 35.4MB/s]
[INFO] Processing subject 1
Filter: 100% 11940/11940 [01:17<00:00, 154.82 examples/s]
[INFO] Subject 1: 1980 samples in full pool
[INFO] Starting latent computation for subj1 with batch_size=16
subj1:  19% 24/124 [01:31<06:18,  3.78s/it][CHECKPOINT] subj1: saved 400 / 1980
subj1:  40% 49/124 [03:07<04:47,  3.84s/it][CHECKPOINT] subj1: saved 800 / 1980
subj1:  60% 74/124 [04:44<03:11,  3.83s/it][CHECKPOINT] subj1: saved 1200 / 1980
subj1:  80% 99/124 [06:21<01:36,  3.88s/it][CHECKPOINT] subj1: saved 1600 / 

## Precompute CLIP embeddings

In [ ]:
!micromamba run -p "{ENV_PATH}" python precompute_clip_embeds.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --batch_size 32 \
  --output_root data/clip_embeds

[INFO] Loading CLIP processor/model...
preprocessor_config.json: 100% 316/316 [00:00<00:00, 53.2kB/s]
config.json: 4.19kB [00:00, 22.1MB/s]
pytorch_model.bin: 100% 605M/605M [00:04<00:00, 122MB/s]
[INFO] Loading full HF pool...
[INFO] Full pool size: 11940
[INFO] Subject 1: 1980 samples
subj1: 100% 62/62 [00:37<00:00,  1.67it/s]
[DONE] subj1 -> data/clip_embeds/luigi-s_EEG_Image_CVPR_ALL_subj/subj1/clip_img_embeds.npy | shape=(1980, 512)
[INFO] Subject 2: 1992 samples
subj2: 100% 63/63 [00:14<00:00,  4.28it/s]
[DONE] subj2 -> data/clip_embeds/luigi-s_EEG_Image_CVPR_ALL_subj/subj2/clip_img_embeds.npy | shape=(1992, 512)
[INFO] Subject 3: 1992 samples
subj3: 100% 63/63 [00:14<00:00,  4.36it/s]
[DONE] subj3 -> data/clip_embeds/luigi-s_EEG_Image_CVPR_ALL_subj/subj3/clip_img_embeds.npy | shape=(1992, 512)
[INFO] Subject 4: 1994 samples
subj4: 100% 63/63 [00:14<00:00,  4.28it/s]
[DONE] subj4 -> data/clip_embeds/luigi-s_EEG_Image_CVPR_ALL_subj/subj4/clip_img_embeds.npy | shape=(1994, 512)
[IN

## Baseline (GWIT reproduction)

### Start training (baseline)

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lorenzosoannini (lorenzosoannini-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit accelerate launch train.py \
  --caption_from_classifier \
  --subject_num=4 \
  --pretrained_model_name_or_path=Manojb/stable-diffusion-2-1-base \
  --output_dir=output/BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
  --dataset_name=luigi-s/EEG_Image_CVPR_ALL_subj \
  --conditioning_image_column=conditioning_image \
  --image_column=image \
  --caption_column=caption \
  --learning_rate=1e-5 \
  --train_batch_size=6 \
  --num_train_epochs=50 \
  --checkpointing_steps=2000 \
  --validation_steps=500 \
  --enable_xformers_memory_efficient_attention \
  --report_to=wandb \
  --tracker_project_name=BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
  --data_root="/content/drive/My Drive/GWIT2/data" \
  --use_precomputed_latents \
  --latents_dir="data/luigi-s_EEG_Image_CVPR_ALL_subj_latents_train_subj4" \
  --drop_coarse_control_prob=0.5 \
  --log_drop_coarse_control \
  #--resume_from_checkpoint=checkpoint-8000

/content
2026-02-23 11:48:32,560 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

tokenizer_config.json: 100% 807/807 [00:00<00:00, 138kB/s]
vocab.json: 1.06MB [00:00, 14.0MB/s]
merges.txt: 525kB [00:00, 42.3MB/s]
special_tokens_map.json: 100% 460/460 [00:00<00:00, 95.3kB/s]
/root/.local/share/mamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
config.json: 100% 613/613 [00:00<00:00, 180kB/s]
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
scheduler_config.js

### Generate images

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit \
  python generate_controlnet.py \
    --controlnet_path output/BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
    --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
    --caption \
    --subject 4 \
    --single_image_for_eval \
    --guess \
    --batch_size 4

/content
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
STO USANDO LA LIBRERIA GIUSTA
model_index.json: 100% 543/543 [00:00<00:00, 105kB/s]
Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]
merges.txt: 0.00B [00:00, ?B/s]

preprocessor_config.json: 100% 342/342 [00:00<00:00, 33.7kB/s]



tokenizer_config.json:   0% 0.00/807 [00:00<?, ?B/s]

config.json:   0% 0.00/613 [00:00<?, ?B/s]



merges.txt: 525kB [00:00, 15.2MB/s]
tokenizer_config.json: 100% 807/807 [00:00<00:00, 38.2kB/s]


scheduler_config.json:   0% 0.00/346 [00:00<?, ?B/s]
config.json: 100% 613/613 [00:00<00:00, 22.9kB/s]
scheduler_config.json: 100% 346/346 [00:00<00:00, 27.5kB/s]
special_tokens_map.json: 100% 460/460 [00:00<00:00, 69.4kB/s]
vocab.json: 1.06MB [00:00, 29.6MB/s]

model.safetensors:   0% 0.00/1.36G [00:00<?, ?B/s]

### Evaluate

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit python evaluation/evaluate.py \
  --controlnet_path output/BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
  --batch_size 32 \
  --limit 4 \
  --guess \
  --GA

/content
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100% 233M/233M [00:01<00:00, 239MB/s]
/root/.local/share/mamba/envs/gwit/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/.local/share/mamba/envs/gwit/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_go

## ZEBRA integration

### Train

In [9]:
ùimport wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lorenzosoannini (lorenzosoannini-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [10]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/loso_subj6_sife_debug_v2 \
  --report_to wandb \
  --tracker_project_name loso_subj6_sife_debug_v2 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --train_batch_size 6 \
  --val_batch_size 4 \
  --max_train_steps 2000 \
  --checkpointing_steps 2000 \
  --checkpoints_total_limit 3 \
  --console_log_every 20 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-5 \
  --eeg_backbone_lr 1e-5 \
  --sife_lr 1e-5 \
  --recon_lr 1e-5 \
  --ssfe_lr 1e-5 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --caption_from_classifier \
  --use_precomputed_latents \
  --latents_dir "/content/gwit_runtime/latents/luigi-s_EEG_Image_CVPR_ALL_subj" \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --grl_lambda_sife 0.1 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 0.1 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 0.5 \
  --max_train_samples_per_subject 400 \
  --max_val_samples_per_subject 1

2026-03-23 15:41:22,650 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
2026-03-23 15:43:02,506 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | 

### Generate

In [ ]:
!micromamba run -p "{ENV_PATH}" python generate_controlnet.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --controlnet_path /content/drive/MyDrive/GWIT2/output/loso_subj6_minicheck \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/loso_subj6_minicheck/generation \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --max_test_samples_per_subject 50 \
  --caption_from_classifier \
  --num_images_per_sample 4 \
  --num_inference_steps 20 \
  --guidance_scale 1.0 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --batch_size 16 \
  --seed 42

### Evaluate

In [ ]:
!micromamba run -p "{ENV_PATH}" python evaluate.py \
  --controlnet_path /content/drive/MyDrive/GWIT2/output/loso_subj6_minicheck/generation \
  --eval_mode grouped \
  --fid_variant all \
  --GA \
  --clip_metrics